In [ ]:
# Общие импорты и локальные данные JERICHON
import os, sys
from pathlib import Path

_HERE = Path(globals().get("__vsc_ipynb_file__", Path.cwd())).expanduser().resolve()
_CANDIDATES = [_HERE, *_HERE.parents, Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next((p for p in dict.fromkeys(_CANDIDATES)
                     if (p / "src" / "data" / "load_data.py").is_file()
                     and (p / "research").is_dir()), None)
if PROJECT_ROOT is None and os.environ.get("JERICHON_PROJECT_ROOT"):
    PROJECT_ROOT = Path(os.environ["JERICHON_PROJECT_ROOT"]).expanduser().resolve()
if PROJECT_ROOT is None or not (PROJECT_ROOT / "src" / "data" / "load_data.py").is_file():
    raise FileNotFoundError("Не найден корень JERICHON с src/data/load_data.py")
sys.path[:0] = [str(PROJECT_ROOT), str(PROJECT_ROOT / "data")]
from src.data.load_data import *
ROOT = PROJECT_ROOT


# JERICHON — Super Candles: группировка признаков и event study v2

**Цель:** проверить источники и семейства Super Candle-признаков без 401 отдельных тестов.

Зафиксированные архитектурные правила из диагностики: Super Candles = только `tradestats`, `obstats`, `orderstats`; `orderbook` отдельно. Не делать простой join по `ts`: использовать существующий causal pipeline с валидированным `available_at <= bar_ts` и backward join. `SYSTIME` нельзя глобально считать UTC или MSK без проверки.

Этот notebook повторно строит локальный causal dataset, сворачивает трансформации признаков в корневые сигналы, формирует source/family scores и объединяет соседние срабатывания в эпизоды. На диск записывается **ровно один CSV**: `SC_02_group_research_summary.csv`. Результаты разведочные, не доказательство торгового преимущества.

## 1. Конфигурация

Используется локальная история AlgoPack, без API-запросов. Порог Dollar Bars должен совпадать с предыдущим исследованием.

In [1]:
from pathlib import Path
import sys, inspect, warnings, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TICKERS = ["SMLT", "GAZP", "MTLR", "VTBR"]
DOLLAR_THRESHOLD = 10_000_000
ROLLING_WINDOW = 500
MIN_HISTORY = 100
ANOMALY_QUANTILES = (0.95, 0.99)
GROUP_QUANTILES = (0.95, 0.99)
EPISODE_GAP_BARS = 3
MAX_CONTROLS_PER_EVENT = 3
RANDOM_STATE = 17
PROJECT_ROOT = ROOT
DATA_ROOT = PROJECT_ROOT / "data"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.builders import jerichon_feature_pipeline as jfp

required_helpers = ("build_dollar_bars_from_trades", "_base", "_build_sc_features", "_choose_availability", "_attach_source")
missing = [x for x in required_helpers if not hasattr(jfp, x)]
if missing:
    raise RuntimeError(f"В установленном jerichon_feature_pipeline.py отсутствуют helpers: {missing}")

print("Project root:", PROJECT_ROOT)
print("History root:", HISTORY_ROOT)
print("Pipeline:", Path(inspect.getfile(jfp)).resolve())
print("Tickers:", TICKERS)
print("Dollar threshold:", f"{DOLLAR_THRESHOLD:,}")


Project root: C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON
History root: C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\storage\history
Pipeline: C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\builders\jerichon_feature_pipeline.py
Tickers: ['SMLT', 'GAZP', 'MTLR', 'VTBR']
Dollar threshold: 10,000,000


## 2. Построение причинного набора

Повторно используем проверенную загрузку и существующий feature pipeline; контракт 73 + 401 = 474 проверяется до анализа.

In [2]:

def resolve_feature_frame(source, frames):
    if source in frames:
        return frames[source]
    matches = [k for k in frames if source.lower() in str(k).lower()]
    if len(matches) == 1:
        return frames[matches[0]]
    raise RuntimeError(f"Не удалось однозначно сопоставить {source}; доступны: {list(frames)}")

def build_ticker_dataset(ticker):
    data, saved_dates = load_local_history(ticker, HISTORY_ROOT)
    bars = jfp.build_dollar_bars_from_trades(data["trades"], threshold=DOLLAR_THRESHOLD)
    if bars is None or bars.empty:
        raise RuntimeError(f"{ticker}: Dollar Bar builder вернул пустой результат.")
    if "bar_ts" not in bars.columns:
        if "close_time" not in bars.columns:
            raise RuntimeError(f"{ticker}: в bars нет bar_ts/close_time; columns={list(bars.columns)}")
        bars["bar_ts"] = pd.to_datetime(bars["close_time"], errors="coerce", utc=True)
    else:
        bars["bar_ts"] = pd.to_datetime(bars["bar_ts"], errors="coerce", utc=True)
    bars = bars.dropna(subset=["bar_ts"]).sort_values("bar_ts").drop_duplicates("bar_ts", keep="last").reset_index(drop=True)

    # The pipeline creates many derived columns by repeated assignment. Its
    # fragmentation PerformanceWarnings are known and are summarized once,
    # rather than flooding the notebook output.
    perf_warning_count = 0
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always", pd.errors.PerformanceWarning)
        base = jfp._base(bars.copy())
    perf_warning_count += sum(issubclass(w.category, pd.errors.PerformanceWarning) for w in captured)
    if isinstance(base, tuple):
        base = base[0]
    if not isinstance(base, pd.DataFrame):
        raise TypeError(f"{ticker}: _base не вернул DataFrame")
    if "bar_ts" not in base.columns:
        if "close_time" in base.columns:
            base = base.rename(columns={"close_time": "bar_ts"})
        elif isinstance(base.index, pd.DatetimeIndex):
            base = base.reset_index().rename(columns={base.index.name or "index": "bar_ts"})
        elif "ts" in base.columns:
            base = base.rename(columns={"ts": "bar_ts"})
        else:
            raise RuntimeError(f"{ticker}: не найден временной ключ в base features")
    base["bar_ts"] = pd.to_datetime(base["bar_ts"], errors="coerce", utc=True)
    base = base.dropna(subset=["bar_ts"]).sort_values("bar_ts").drop_duplicates("bar_ts", keep="last")

    bars_sc = bars.copy()
    if "ts" not in bars_sc.columns:
        bars_sc["ts"] = bars_sc["bar_ts"]
    if "CLOSE" not in bars_sc.columns:
        close_col = next((c for c in bars_sc.columns if str(c).lower() == "close"), None)
        if close_col is None:
            raise RuntimeError(f"{ticker}: нет CLOSE/close в Dollar Bars")
        bars_sc["CLOSE"] = bars_sc[close_col]
    bars_sc["ts"] = pd.to_datetime(bars_sc["ts"], errors="coerce", utc=True)
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always", pd.errors.PerformanceWarning)
        sc_result = jfp._build_sc_features(
            data,
            bars_sc.dropna(subset=["ts"]).sort_values("ts").reset_index(drop=True),
        )
    perf_warning_count += sum(issubclass(w.category, pd.errors.PerformanceWarning) for w in captured)
    feature_frames = sc_result[0] if isinstance(sc_result, tuple) else sc_result
    if not isinstance(feature_frames, dict):
        raise TypeError(f"{ticker}: _build_sc_features должен вернуть dict")

    causal = bars.copy()
    for source in ("tradestats", "obstats", "orderstats"):
        # Validate source availability using the pipeline's established helper.
        checked = jfp._choose_availability(data[source].copy(), source)
        if isinstance(checked, tuple):
            checked = checked[0]
        if checked is None or len(checked) == 0:
            raise RuntimeError(f"{ticker}: availability validation вернула пустой {source}")
        ff = resolve_feature_frame(source, feature_frames)
        attached = jfp._attach_source(causal, ff, data[source], source)
        if isinstance(attached, pd.DataFrame):
            causal = attached

    base_cols_raw = [c for c in base.columns if c != "bar_ts" and pd.api.types.is_numeric_dtype(base[c])]
    if len(base_cols_raw) != 73:
        raise RuntimeError(f"{ticker}: ожидалось 73 base features, найдено {len(base_cols_raw)}")
    base_join = base[["bar_ts"] + base_cols_raw].copy()
    base_join = base_join.rename(columns={c: f"base__{c}" for c in base_cols_raw})
    causal["bar_ts"] = pd.to_datetime(causal["bar_ts"], errors="coerce", utc=True)
    causal = causal.merge(base_join, on="bar_ts", how="left", validate="one_to_one")

    provenance = [c for c in causal.columns if c.endswith("__event_ts") or c.endswith("__available_at")]
    prefixes = ("tradestats__", "obstats__", "orderstats__")
    sc_cols = [c for c in causal.columns if c.startswith(prefixes) and c not in provenance and pd.api.types.is_numeric_dtype(causal[c])]
    # IMPORTANT: test the dtype of the SERIES causal[c], not the column-name
    # string c. Testing c silently dropped all 73 base features.
    base_feature_cols = [
        c for c in causal.columns
        if c.startswith("base__") and pd.api.types.is_numeric_dtype(causal[c])
    ]
    feature_cols = base_feature_cols + sc_cols

    # Validate each block separately so a mismatch reports exactly which block
    # is absent; don't silently skip a ticker or hide the root cause.
    if len(base_feature_cols) != len(base_cols_raw):
        raise RuntimeError(
            f"{ticker}: после merge сохранилось {len(base_feature_cols)} из "
            f"{len(base_cols_raw)} базовых признаков"
        )
    if not sc_cols:
        raise RuntimeError(
            f"{ticker}: после causal attach не найдено числовых SC-признаков. "
            f"Колонки итогового frame: {list(causal.columns)}"
        )
    if len(feature_cols) != 474:
        raise RuntimeError(
            f"{ticker}: состав признаков отличается от ожидаемого: "
            f"base={len(base_feature_cols)} (ожидалось 73), "
            f"SC={len(sc_cols)} (ожидалось 401), total={len(feature_cols)}. "
            "Тикер не будет молча отброшен: проверь названия колонок в pipeline."
        )
    causal = causal.sort_values("bar_ts").drop_duplicates("bar_ts", keep="last").reset_index(drop=True)
    # Keep bar fields for forward path analysis, plus features and provenance.
    if perf_warning_count:
        print(f"{ticker}: подавлено предупреждений о фрагментации DataFrame: {perf_warning_count}")
    return {
        "ticker": ticker, "data": data, "dates": saved_dates, "bars": bars,
        "dataset": causal, "feature_cols": feature_cols, "base_cols": base_feature_cols,
        "sc_cols": sc_cols, "provenance": provenance,
    }

datasets = {}
load_rows = []
load_errors = []
for ticker in TICKERS:
    try:
        obj = build_ticker_dataset(ticker)
        datasets[ticker] = obj
        load_rows.append({
            "ticker": ticker, "saved_dates": len(obj["dates"]),
            "date_start": obj["dates"][0], "date_end": obj["dates"][-1],
            "dollar_bars": len(obj["bars"]), "base_features": len(obj["base_cols"]),
            "supercandle_features": len(obj["sc_cols"]), "features_total": len(obj["feature_cols"]),
        })
        print(
            f"OK {ticker}: bars={len(obj['bars']):,}; "
            f"base={len(obj['base_cols'])}; SC={len(obj['sc_cols'])}; "
            f"dates={obj['dates'][0]} → {obj['dates'][-1]}"
        )
    except Exception as e:
        message = f"{ticker}: {type(e).__name__}: {e}"
        load_errors.append(message)
        print(f"ERROR {message}")
display(pd.DataFrame(load_rows))
if load_errors:
    print("\nОшибки загрузки (кратко):")
    for message in load_errors:
        print(" -", message)
if not datasets:
    raise RuntimeError(
        "Не удалось подготовить данные ни для одного тикера. Причины выше: "
        + " | ".join(load_errors)
    )


SMLT: подавлено предупреждений о фрагментации DataFrame: 74
OK SMLT: bars=2,672; base=73; SC=401; dates=2026-09-25 → 2026-10-20
GAZP: подавлено предупреждений о фрагментации DataFrame: 74
OK GAZP: bars=9,993; base=73; SC=401; dates=2026-09-24 → 2026-10-20
MTLR: подавлено предупреждений о фрагментации DataFrame: 74
OK MTLR: bars=1,492; base=73; SC=401; dates=2026-08-28 → 2026-10-20
VTBR: подавлено предупреждений о фрагментации DataFrame: 74
OK VTBR: bars=9,482; base=73; SC=401; dates=2026-08-28 → 2026-10-20


,ticker,saved_dates,date_start,date_end,dollar_bars,base_features,supercandle_features,features_total
0,SMLT,14,2026-09-25,2026-10-20,2672,73,401,474
1,GAZP,15,2026-09-24,2026-10-20,9993,73,401,474
2,MTLR,33,2026-08-28,2026-10-20,1492,73,401,474
3,VTBR,32,2026-08-28,2026-10-20,9482,73,401,474


## 3. Свернуть 401 столбец в корневые метрики и смысловые семьи

Например, `x`, `x_ma3`, `x_z12`, `x_chg`, `x_ret` относятся к одному корневому сигналу. Семьи по названиям эвристические и нужны только для первого анализа.

In [3]:
import math, re
from collections import defaultdict

TRANSFORM_RE = re.compile(r"_(?:ma\d+|z\d+|chg|ret)$", re.IGNORECASE)
def root_feature_name(col):
    source, metric = col.split("__", 1)
    while True:
        new_metric = TRANSFORM_RE.sub("", metric)
        if new_metric == metric: break
        metric = new_metric
    return f"{source}__{metric}"
def transform_complexity(col):
    return len(TRANSFORM_RE.findall(col.split("__", 1)[1]))
def semantic_family(root_col):
    m = root_col.split("__", 1)[1].lower()
    if any(k in m for k in ("imb", "imbalance", "bid", "ask", "buy", "sell", "aggress", "b_s", "share")): return "imbalance_pressure"
    if any(k in m for k in ("spread", "liquidity", "cost", "vwap", "px_", "price", "mid", "microprice", "bps")): return "liquidity_price"
    if any(k in m for k in ("volume", "value", "turnover", "trade", "count", "qty", "notional", "amount")): return "activity_volume"
    if any(k in m for k in ("order", "depth", "level", "queue", "cancel", "add", "modify")): return "orders_depth"
    return "other"

# Inventory is shared across tickers; the loader separately validates 73 + 401 = 474 per ticker.
all_sc_cols = datasets[next(iter(datasets))]["sc_cols"]
root_members = defaultdict(list)
for col in all_sc_cols: root_members[root_feature_name(col)].append(col)
root_rows = []
for root, members in sorted(root_members.items()):
    source = root.split("__", 1)[0]
    chosen = root if root in members else sorted(members, key=lambda c:(transform_complexity(c),len(c),c))[0]
    root_rows.append({"root":root,"source":source,"family":semantic_family(root),"chosen_feature":chosen,"variants":len(members),"is_raw_root":chosen==root})
root_map = pd.DataFrame(root_rows)
print(f"401 SC columns collapsed into {root_map.root.nunique()} root metrics")
print("Root metrics by source/family:")
print(root_map.groupby(["source","family"]).size().rename("roots").to_string())
print("\nTransformation variants per root:")
print(root_map.variants.value_counts().sort_index().rename_axis("variants").to_string())


401 SC columns collapsed into 51 root metrics
Root metrics by source/family:
source      family            
obstats     activity_volume        2
            imbalance_pressure     7
            liquidity_price        7
orderstats  activity_volume        4
            imbalance_pressure    13
            liquidity_price        2
            orders_depth           2
tradestats  activity_volume        2
            imbalance_pressure    10
            other                  2

Transformation variants per root:
variants
1     16
11    35


## 4. Причинные групповые scores

Для каждого корневого признака отклонение от rolling median нормируется на прошлый 95-й перцентиль отклонений. Все окна сдвинуты на один бар. Затем групповой score — среднее верхних 20% root-scores; событие — превышение прошлого 95-го или 99-го перцентиля группового score.

In [4]:
def find_ohlc_columns(bars):
    lower = {str(c).lower():c for c in bars.columns}
    out = {k:next((lower[x] for x in names if x in lower),None) for k,names in {
        "open":("open","o"),"high":("high","h"),"low":("low","l"),"close":("close","c")}.items()}
    missing = [k for k,v in out.items() if v is None]
    if missing: raise RuntimeError(f"OHLC columns missing {missing}; columns={list(bars.columns)}")
    return out

def root_scores_for_ticker(obj):
    ds = obj["dataset"].reset_index(drop=True)
    out = {}
    for _, r in root_map.iterrows():
        col = r["chosen_feature"]
        if col not in ds: continue
        x = pd.to_numeric(ds[col],errors="coerce").replace([np.inf,-np.inf],np.nan)
        med = x.rolling(ROLLING_WINDOW,min_periods=MIN_HISTORY).median().shift(1)
        dev = (x-med).abs()
        scale = dev.rolling(ROLLING_WINDOW,min_periods=MIN_HISTORY).quantile(.95).shift(1)
        out[r["root"]] = dev / scale.where(scale>0)
    return pd.DataFrame(out)

def top_fraction_mean(frame, fraction=.20):
    if frame.shape[1]==0: return pd.Series(np.nan,index=frame.index)
    a=frame.to_numpy(dtype=float).copy(); a[~np.isfinite(a)]=-np.inf; a.sort(axis=1)
    k=max(1,int(math.ceil(frame.shape[1]*fraction))); top=a[:,-k:]
    ok=np.isfinite(top); sums=np.where(ok,top,0).sum(axis=1); counts=ok.sum(axis=1)
    return pd.Series(np.divide(sums,counts,out=np.full(len(frame),np.nan),where=counts>0),index=frame.index)

def group_scores_for_ticker(obj):
    roots=root_scores_for_ticker(obj); meta=root_map.set_index("root"); groups={}
    for source in ("tradestats","obstats","orderstats"):
        cols=[c for c in roots if meta.loc[c,"source"]==source]
        if cols: groups[f"source::{source}"]=top_fraction_mean(roots[cols])
    for (source,family),part in root_map.groupby(["source","family"]):
        cols=[c for c in part.root.tolist() if c in roots]
        if len(cols)>=2: groups[f"source_family::{source}::{family}"]=top_fraction_mean(roots[cols])
    scores=pd.DataFrame(groups)
    flags={}; thresholds={}
    for col in scores:
        for q in GROUP_QUANTILES:
            th=scores[col].rolling(ROLLING_WINDOW,min_periods=MIN_HISTORY).quantile(q).shift(1)
            thresholds[(col,q)]=th; flags[(col,q)]=scores[col].gt(th)&th.notna()&scores[col].notna()
    return scores,thresholds,flags

group_results={}
for ticker,obj in datasets.items():
    scores,thresholds,flags=group_scores_for_ticker(obj)
    group_results[ticker]={"scores":scores,"thresholds":thresholds,"flags":flags}
    print(f"{ticker}: {scores.shape[1]} source/family groups; {len(scores):,} bars")


SMLT: 13 source/family groups; 2,672 bars
GAZP: 13 source/family groups; 9,993 bars
MTLR: 13 source/family groups; 1,492 bars
VTBR: 13 source/family groups; 9,482 bars


## 5. Что именно проверяем в этом исследовании

**Главная идея:** не считать GAZP «лучше» MTLR только потому, что у GAZP больше групп с небольшим отклонением. Для потенциальной торговли важны и **размер движения**, и **частота**, и **стабильность** этого эффекта.

Для каждого события сравниваем будущий путь цены с контрольными барами, подобранными по прошлой волатильности, направлению недавнего тренда и близости во времени.

Основные показатели на горизонтах 5 и 20 Dollar Bars (дополнительно 1, 3 и 10):

- **Абсолютная доходность закрытия** — модуль изменения цены от закрытия сигнального бара до закрытия через H баров. Она не зависит от направления.
- **Максимальный ход вверх / вниз** — наибольший рост от цены сигнала и наибольшее падение от неё внутри следующих H баров.
- **Максимальный ход в любую сторону** — большее из абсолютного хода вверх и вниз. Это особенно важно для двусторонней торговли.
- **Диапазон будущего движения** — максимум High минус минимум Low на горизонте, в процентах от цены сигнала.
- **MFE (Maximum Favorable Excursion, максимальное благоприятное движение)** — насколько далеко цена успела пройти в выгодную сторону. Для лонга это ход вверх; для шорта — ход вниз, взятый со знаком плюс.
- **MAE (Maximum Adverse Excursion, максимальное неблагоприятное движение)** — насколько далеко цена уходила против позиции. Для лонга это ход вниз (отрицательное значение); для шорта — ход вверх со знаком минус.

MFE/MAE здесь описывают возможный путь цены, **а не готовый торговый результат**: без правила входа/выхода нельзя считать их PnL.

### Что считается более убедительным результатом

1. MTLR может оказаться интереснее, даже если срабатываний меньше, если при срабатывании заметно растёт вероятность хода на ±1.5–2%.
2. Отдельно оцениваем вверх и вниз; положительный signed return не является обязательным условием.
3. Сравниваем событие с контролем, а не только смотрим на большой ход после события: в обычном рынке такой ход тоже мог бы произойти.
4. Учитываем число независимых эпизодов, интервалы неопределённости и поправку на множественные проверки.
5. Это event study (исследование событий), не бэктест стратегии и не обещание, что движение можно было исполнить с прибылью.

In [5]:
# Основные параметры исследования
HORIZONS = (1, 3, 5, 10, 20, 30)
PRIMARY_HORIZONS = (5, 25)
MAX_HORIZON = max(HORIZONS)

# Объединяем сигналы, если между ними не прошло MAX_HORIZON баров:
# окна будущих исходов не должны многократно считаться независимыми.
EPISODE_GAP_BARS = MAX_HORIZON + 1

# Контроли выбираются только локально во времени и только из похожего режима.
LOCAL_MATCH_RADIUS = 250
MAX_CONTROLS_PER_EVENT = 3
VOL_LOOKBACK = 50
TREND_LOOKBACK = 20

BOOTSTRAP_REPS = 1000
BOOTSTRAP_BLOCK_EPISODES = 4
RANDOM_STATE = 31

# Практически заметные ходы. Это пороги для оценки частоты, не торговые цели.
MOVE_THRESHOLDS_PCT = (0.5, 1.0, 1.5, 2.0)

print("Горизонты:", HORIZONS)
print("Локальный радиус сопоставления:", LOCAL_MATCH_RADIUS, "баров")
print("Минимальный промежуток между эпизодами:", EPISODE_GAP_BARS, "баров")

Горизонты: (1, 3, 5, 10, 20, 30)
Локальный радиус сопоставления: 250 баров
Минимальный промежуток между эпизодами: 31 баров


## 6. Метрики будущего пути и независимые эпизоды

Сигнал доступен на закрытии соответствующего Dollar Bar. Метрики ниже начинаются **со следующего бара**, чтобы не включать диапазон самого сигнального бара в будущий исход.

Важное ограничение: OHLC не показывает порядок движения внутри свечи. Поэтому эти метрики нельзя трактовать как точную последовательность срабатывания take-profit/stop-loss.

In [6]:
def find_ohlc_columns(bars):
    lower = {str(c).lower(): c for c in bars.columns}
    out = {
        "open": next((lower[x] for x in ("open", "o") if x in lower), None),
        "high": next((lower[x] for x in ("high", "h") if x in lower), None),
        "low": next((lower[x] for x in ("low", "l") if x in lower), None),
        "close": next((lower[x] for x in ("close", "c") if x in lower), None),
    }
    missing = [k for k, v in out.items() if v is None]
    if missing:
        raise RuntimeError(f"OHLC columns missing {missing}; columns={list(bars.columns)}")
    return out

def make_path_metrics(close, high, low, i):
    p0 = close[i]
    if not np.isfinite(p0) or p0 == 0:
        return {}
    out = {}
    for h in HORIZONS:
        j = i + h
        if j >= len(close):
            continue
        hi = high[i+1:j+1]
        lo = low[i+1:j+1]
        hi = hi[np.isfinite(hi)]
        lo = lo[np.isfinite(lo)]
        if not len(hi) or not len(lo):
            continue
        ret = (close[j] / p0 - 1) * 100 if np.isfinite(close[j]) else np.nan
        up = (np.max(hi) / p0 - 1) * 100
        down = (np.min(lo) / p0 - 1) * 100  # обычно отрицательное число
        out[f"ret_{h}"] = ret
        out[f"abs_ret_{h}"] = abs(ret) if np.isfinite(ret) else np.nan
        out[f"max_up_{h}"] = up
        out[f"max_down_{h}"] = down
        out[f"max_abs_move_{h}"] = max(abs(up), abs(down))
        out[f"range_{h}"] = (np.max(hi) - np.min(lo)) / p0 * 100
        # Для long: MFE = ход вверх; MAE = ход вниз.
        out[f"long_mfe_{h}"] = up
        out[f"long_mae_{h}"] = down
        # Для short: MFE = падение цены; MAE = рост цены.
        out[f"short_mfe_{h}"] = -down
        out[f"short_mae_{h}"] = -up
        for threshold in MOVE_THRESHOLDS_PCT:
            tag = str(threshold).replace(".", "p")
            out[f"up_ge_{tag}_{h}"] = float(up >= threshold)
            out[f"down_ge_{tag}_{h}"] = float(down <= -threshold)
            out[f"either_ge_{tag}_{h}"] = float(max(abs(up), abs(down)) >= threshold)
    return out

def make_context(bars):
    c = find_ohlc_columns(bars)
    close = pd.to_numeric(bars[c["close"]], errors="coerce").to_numpy(float)
    high = pd.to_numeric(bars[c["high"]], errors="coerce").to_numpy(float)
    low = pd.to_numeric(bars[c["low"]], errors="coerce").to_numpy(float)
    ret = pd.Series(close).pct_change()
    # Все признаки режима сдвинуты назад: они используют только данные до бара события.
    vol = ret.rolling(VOL_LOOKBACK, min_periods=max(20, VOL_LOOKBACK // 2)).std().shift(1)
    trend = (pd.Series(close).pct_change(TREND_LOOKBACK) * 100).shift(1)
    # Причинные квинтили волатильности: пороги строятся только на прошлых значениях.
    # Это важно даже для подбора контролей — не используем распределение будущей истории.
    vol_bucket = pd.Series(np.nan, index=vol.index, dtype=float)
    qcuts = {
        q: vol.rolling(ROLLING_WINDOW, min_periods=MIN_HISTORY).quantile(q).shift(1)
        for q in (0.2, 0.4, 0.6, 0.8)
    }
    valid = vol.notna() & qcuts[0.2].notna() & qcuts[0.4].notna() & qcuts[0.6].notna() & qcuts[0.8].notna()
    vol_bucket.loc[valid] = (
        (vol.loc[valid] > qcuts[0.2].loc[valid]).astype(int)
        + (vol.loc[valid] > qcuts[0.4].loc[valid]).astype(int)
        + (vol.loc[valid] > qcuts[0.6].loc[valid]).astype(int)
        + (vol.loc[valid] > qcuts[0.8].loc[valid]).astype(int)
    ).to_numpy()
    trend_bucket = pd.Series(
        np.select([trend < -0.15, trend > 0.15], [-1, 1], default=0),
        index=trend.index
    ).where(trend.notna())
    return c, close, high, low, vol, vol_bucket, trend_bucket

## 7. Контрольные бары: похожий режим и близкое время

Контроль должен быть не только похож по волатильности, но и близок по времени к событию. Иначе, например, сигнал из спокойного месяца может сравниваться с контролем из кризисного периода.

Для каждого эпизода ищем контрольные бары в пределах `LOCAL_MATCH_RADIUS`, совпадающие по квинтилю прошлой волатильности и категории недавнего тренда. Из контрольных кандидатов исключаются бары, окна будущих исходов которых пересекаются с эпизодами сигнала. Если сопоставимых контролей нет, событие не участвует в парном сравнении; это отражается в количестве эпизодов с контролями.

In [10]:
def episode_starts(mask, min_gap=EPISODE_GAP_BARS):
    starts = []
    last = -10**9
    for i in np.flatnonzero(np.asarray(mask, dtype=bool)):
        if int(i) - last >= min_gap:
            starts.append(int(i))
            last = int(i)
    return starts

def make_event_contamination_mask(n, event_starts):
    # Исключаем контрольные бары, чьё будущее окно пересекается с окном любого события.
    contaminated = np.zeros(n, dtype=bool)
    for e in event_starts:
        lo = max(0, e - MAX_HORIZON)
        hi = min(n, e + 2 * MAX_HORIZON + 1)
        contaminated[lo:hi] = True
    return contaminated

def match_controls(event_i, candidates, vol_bucket, trend_bucket, event_starts, n, rng, contaminated):
    left = max(MIN_HISTORY, event_i - LOCAL_MATCH_RADIUS)
    right = min(n - MAX_HORIZON - 1, event_i + LOCAL_MATCH_RADIUS)
    vb = vol_bucket.iloc[event_i]
    tb = trend_bucket.iloc[event_i]
    if pd.isna(vb) or pd.isna(tb):
        return []
    pool = []
    for j in range(left, right + 1):
        if abs(j - event_i) <= MAX_HORIZON:
            continue
        if pd.isna(vol_bucket.iloc[j]) or pd.isna(trend_bucket.iloc[j]):
            continue
        if int(vol_bucket.iloc[j]) != int(vb) or int(trend_bucket.iloc[j]) != int(tb):
            continue
        if contaminated[j]:
            continue
        pool.append(j)
    if not pool:
        return []
    # Сначала ближайшие по времени; случайный выбор только среди равных по дистанции.
    pool = sorted(pool, key=lambda j: abs(j-event_i))
    nearest_distance = abs(pool[0]-event_i)
    near_ties = [j for j in pool if abs(j-event_i) <= nearest_distance + 5]
    take = min(MAX_CONTROLS_PER_EVENT, len(near_ties))
    return sorted(rng.choice(near_ties, size=take, replace=False).tolist())

def analyze_group_robust(ticker, obj, group_key, q, flag):
    bars = obj["bars"].reset_index(drop=True)
    _, close, high, low, vol, vol_bucket, trend_bucket = make_context(bars)
    n = len(close)
    valid_end = n - MAX_HORIZON - 1
    raw_mask = np.asarray(flag, dtype=bool)
    raw_mask[valid_end:] = False
    starts = episode_starts(raw_mask)
    if not starts:
        return []

    stable_seed = RANDOM_STATE + sum(ord(ch) for ch in f"{ticker}|{group_key}|{q}")
    rng = np.random.default_rng(stable_seed)
    rows = []
    group_type, group = group_key.split("::", 1)
    contaminated = make_event_contamination_mask(n, starts)
    for ei in starts:
        ctr_idx = match_controls(ei, range(n), vol_bucket, trend_bucket, starts, n, rng, contaminated)
        if not ctr_idx:
            continue
        em = make_path_metrics(close, high, low, ei)
        cms = [make_path_metrics(close, high, low, j) for j in ctr_idx]
        if not em:
            continue
        row = {
            "ticker": ticker, "group_type": group_type, "group": group, "quantile": q,
            "event_idx": ei, "event_bar_ts": bars.iloc[ei].get("bar_ts", pd.NaT),
            "bars": n, "raw_event_bars": int(raw_mask.sum()),
            "event_bar_pct": 100 * raw_mask.sum() / max(n, 1),
            "episodes": len(starts), "controls_n": len(ctr_idx),
            "control_indices": "|".join(map(str, ctr_idx)),
        }
        metric_names = list(em.keys())
        for metric in metric_names:
            vals = np.array([m.get(metric, np.nan) for m in cms], dtype=float)
            vals = vals[np.isfinite(vals)]
            row[f"event_{metric}"] = em[metric]
            row[f"control_{metric}"] = float(vals.mean()) if len(vals) else np.nan
            row[f"diff_{metric}"] = em[metric] - float(vals.mean()) if len(vals) else np.nan
        rows.append(row)
    return rows

event_rows = []
for ticker, obj in datasets.items():
    gr = group_results[ticker]
    for (group_key, q), flag in gr["flags"].items():
        event_rows.extend(analyze_group_robust(ticker, obj, group_key, q, flag))
    print(f"{ticker}: matched episodes so far = {sum(r['ticker']==ticker for r in event_rows):,}")

event_level = pd.DataFrame(event_rows)
if event_level.empty:
    raise RuntimeError(
        "Не найдено событий с локальными контролями. Проверь длину истории, "
        "LOCAL_MATCH_RADIUS и достаточность совпадающих режимов."
    )
print("Matched event episodes:", len(event_level))
display(event_level.head())

SMLT: matched episodes so far = 323
GAZP: matched episodes so far = 1,092
MTLR: matched episodes so far = 168
VTBR: matched episodes so far = 1,200
Matched event episodes: 2783


,ticker,group_type,group,quantile,event_idx,event_bar_ts,bars,raw_event_bars,event_bar_pct,episodes,...,diff_either_ge_1p5_30,event_up_ge_2p0_30,control_up_ge_2p0_30,diff_up_ge_2p0_30,event_down_ge_2p0_30,control_down_ge_2p0_30,diff_down_ge_2p0_30,event_either_ge_2p0_30,control_either_ge_2p0_30,diff_either_ge_2p0_30
0,SMLT,source,tradestats,0.95,316,2026-09-29 05:49:34+00:00,2672,117,4.378743,19,...,0.0,0.0,0.0,0.0,0.0,0.666667,-0.666667,0.0,0.666667,-0.666667
1,SMLT,source,tradestats,0.95,372,2026-09-29 07:55:17+00:00,2672,117,4.378743,19,...,-1.0,0.0,0.0,0.0,0.0,0.333333,-0.333333,0.0,0.333333,-0.333333
2,SMLT,source,tradestats,0.95,569,2026-09-29 13:40:19+00:00,2672,117,4.378743,19,...,0.0,0.0,0.0,0.0,1.0,0.000000,1.000000,1.0,0.000000,1.000000
3,SMLT,source,tradestats,0.95,1350,2026-10-05 06:47:07+00:00,2672,117,4.378743,19,...,-1.0,0.0,0.0,0.0,0.0,1.000000,-1.000000,0.0,1.000000,-1.000000
4,SMLT,source,tradestats,0.95,1723,2026-10-07 13:19:07+00:00,2672,117,4.378743,19,...,1.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000


## 8. Интервалы неопределённости и поправка на множественные проверки

Обычный независимый bootstrap здесь был бы слишком оптимистичен: соседние события могут быть связаны. Поэтому используем **блочный bootstrap по последовательности эпизодов** — пересэмплируем небольшие соседние блоки разностей «событие минус контроль». Это лучше учитывает локальную зависимость, но всё равно остаётся приближением.

`p_value_block` — двусторонняя bootstrap-оценка против нулевой средней разности. `q_value_BH` — поправка Benjamini–Hochberg на множественные проверки внутри каждого показателя и горизонта. Она помогает уменьшить число ложноположительных находок, когда одновременно проверяется много групп.

Не трактуй один `q_value_BH < 0.05` как доказательство торгового преимущества: важны размер эффекта, число эпизодов, устойчивость на новых датах и исполнимость сделки.

In [11]:
def moving_block_bootstrap_mean(x, reps=BOOTSTRAP_REPS, block_len=BOOTSTRAP_BLOCK_EPISODES, seed=0):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n < 8:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    block_len = max(1, min(int(block_len), n))
    n_blocks = int(np.ceil(n / block_len))
    # Percentile CI on observed values; null distribution is centered at zero.
    boot = np.empty(reps)
    centered = x - x.mean()
    null = np.empty(reps)
    starts = np.arange(n)
    for b in range(reps):
        chunks = []
        null_chunks = []
        for _ in range(n_blocks):
            s = int(rng.choice(starts))
            idx = (s + np.arange(block_len)) % n
            chunks.extend(x[idx].tolist())
            null_chunks.extend(centered[idx].tolist())
        boot[b] = np.mean(chunks[:n])
        null[b] = np.mean(null_chunks[:n])
    lo, hi = np.quantile(boot, [0.025, 0.975])
    p = (1 + np.sum(np.abs(null) >= abs(x.mean()))) / (reps + 1)
    return float(lo), float(hi), float(p)

def benjamini_hochberg(pvalues):
    p = np.asarray(pvalues, dtype=float)
    out = np.full(len(p), np.nan)
    valid = np.flatnonzero(np.isfinite(p))
    if not len(valid):
        return out
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    m = len(ranked)
    q = ranked * m / np.arange(1, m + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    vals = np.empty(m)
    vals[order] = np.minimum(q, 1.0)
    out[valid] = vals
    return out

# Keep a concise output: one row per ticker × group × threshold × horizon.
# Primary metrics are direction-neutral and two-sided; signed return is secondary.
PRIMARY_METRICS = ("abs_ret", "max_abs_move", "max_up", "max_down", "range", "ret")
summary_rows = []
for (ticker, group_type, group, q), g in event_level.groupby(
    ["ticker", "group_type", "group", "quantile"], dropna=False
):
    for h in HORIZONS:
        for metric in PRIMARY_METRICS:
            col = f"diff_{metric}_{h}"
            if col not in g:
                continue
            x = pd.to_numeric(g[col], errors="coerce").dropna().to_numpy(float)
            if not len(x):
                continue
            seed = RANDOM_STATE + int(q*1000) + h + sum(map(ord, f"{ticker}|{group}|{metric}"))
            lo, hi, p = moving_block_bootstrap_mean(x, seed=seed)
            evcol, ccol = f"event_{metric}_{h}", f"control_{metric}_{h}"
            summary_rows.append({
                "ticker": ticker, "group_type": group_type, "group": group,
                "quantile": q, "horizon_bars": h, "metric": metric,
                "bars": int(g.bars.iloc[0]), "raw_event_bars": int(g.raw_event_bars.iloc[0]),
                "event_bar_pct": float(g.event_bar_pct.iloc[0]),
                "episodes_total": int(g.episodes.iloc[0]), "episodes_matched": int(len(x)),
                "mean_event": float(pd.to_numeric(g[evcol], errors="coerce").mean()),
                "mean_control": float(pd.to_numeric(g[ccol], errors="coerce").mean()),
                "mean_diff": float(np.mean(x)), "median_diff": float(np.median(x)),
                "ci95_low": lo, "ci95_high": hi, "p_value_block": p,
                "ci_excludes_zero": bool(np.isfinite(lo) and np.isfinite(hi) and (lo > 0 or hi < 0)),
            })

summary = pd.DataFrame(summary_rows)
if summary.empty:
    raise RuntimeError("Не получилось собрать сводку метрик.")
summary["q_value_BH"] = np.nan
for _, idx in summary.groupby(["metric", "horizon_bars"]).groups.items():
    summary.loc[idx, "q_value_BH"] = benjamini_hochberg(summary.loc[idx, "p_value_block"].to_numpy())
summary["fdr_5pct"] = summary["q_value_BH"] < 0.05
summary = summary.sort_values(
    ["ticker", "metric", "group_type", "group", "quantile", "horizon_bars"]
).reset_index(drop=True)

OUTPUT_FILE = Path("research/supercandles/mark_3/results/SC_03_robust_event_study_summary.csv"); OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUTPUT_FILE, index=False)
print(f"Сохранён один компактный CSV: {OUTPUT_FILE.resolve()} ({len(summary):,} строк)")
display(summary.head(20))

Сохранён один компактный CSV: C:\Users\Сергей Кравцов\SC_03_robust_event_study_summary.csv (3,744 строк)


,ticker,group_type,group,quantile,horizon_bars,metric,bars,raw_event_bars,event_bar_pct,episodes_total,...,mean_event,mean_control,mean_diff,median_diff,ci95_low,ci95_high,p_value_block,ci_excludes_zero,q_value_BH,fdr_5pct
0,GAZP,source,obstats,0.95,1,abs_ret,9993,544,5.443811,55,...,0.089007,0.053747,0.035260,-0.004940,-0.009999,0.096379,0.215784,False,0.568462,False
1,GAZP,source,obstats,0.95,3,abs_ret,9993,544,5.443811,55,...,0.140096,0.119030,0.021066,0.019603,-0.030830,0.079731,0.445554,False,0.811596,False
2,GAZP,source,obstats,0.95,5,abs_ret,9993,544,5.443811,55,...,0.155970,0.137004,0.018966,-0.014026,-0.035632,0.069006,0.505495,False,0.812756,False
3,GAZP,source,obstats,0.95,10,abs_ret,9993,544,5.443811,55,...,0.230368,0.145958,0.084410,0.061399,0.018914,0.145964,0.012987,True,0.266234,False
4,GAZP,source,obstats,0.95,20,abs_ret,9993,544,5.443811,55,...,0.305587,0.228882,0.076704,0.050800,0.013908,0.136559,0.012987,True,0.133117,False
5,GAZP,source,obstats,0.95,30,abs_ret,9993,544,5.443811,55,...,0.374422,0.313224,0.061197,0.067623,-0.036247,0.151036,0.227772,False,0.640889,False
6,GAZP,source,obstats,0.99,1,abs_ret,9993,128,1.280897,25,...,0.107750,0.057835,0.049915,-0.013304,-0.023144,0.148721,0.275724,False,0.628039,False
7,GAZP,source,obstats,0.99,3,abs_ret,9993,128,1.280897,25,...,0.134582,0.124005,0.010577,0.008007,-0.067624,0.109198,0.814186,False,0.925568,False
8,GAZP,source,obstats,0.99,5,abs_ret,9993,128,1.280897,25,...,0.422257,0.142924,0.279333,-0.012572,-0.035300,0.804829,0.215784,False,0.628842,False
9,GAZP,source,obstats,0.99,10,abs_ret,9993,128,1.280897,25,...,0.550543,0.147762,0.402781,0.121662,0.111402,0.880252,0.060939,True,0.453700,False


## 9. Вероятность крупного хода в обе стороны

Здесь сравнивается доля будущих окон, в которых цена успела пройти заданный порог вверх, вниз или хотя бы в одну сторону. Например, `either_ge_2p_20` означает, что за 20 баров максимальный ход хотя бы в одну сторону достиг 2%.

Для каждого события разность вероятностей рассчитывается относительно среднего контрольного значения. Это **процентные пункты вероятности**, не доходность и не PnL.

In [9]:
tail_rows = []
for (ticker, group_type, group, q), g in event_level.groupby(
    ["ticker", "group_type", "group", "quantile"], dropna=False
):
    for h in HORIZONS:
        for threshold in MOVE_THRESHOLDS_PCT:
            tag = str(threshold).replace(".", "p")
            for side in ("up", "down", "either"):
                metric = f"{side}_ge_{tag}_{h}"
                evcol, ccol, dcol = f"event_{metric}", f"control_{metric}", f"diff_{metric}"
                if dcol not in g:
                    continue
                x = pd.to_numeric(g[dcol], errors="coerce").dropna().to_numpy(float)
                if not len(x):
                    continue
                lo, hi, p = moving_block_bootstrap_mean(
                    x, seed=RANDOM_STATE + h + int(threshold*100) + sum(map(ord, f"{ticker}|{group}|{side}|{q}"))
                )
                tail_rows.append({
                    "ticker": ticker, "group_type": group_type, "group": group, "quantile": q,
                    "horizon_bars": h, "threshold_pct": threshold, "side": side,
                    "episodes_matched": len(x),
                    "event_probability_pct": 100*pd.to_numeric(g[evcol], errors="coerce").mean(),
                    "control_probability_pct": 100*pd.to_numeric(g[ccol], errors="coerce").mean(),
                    "probability_diff_pp": 100*float(np.mean(x)),
                    "ci95_low_pp": 100*lo if np.isfinite(lo) else np.nan,
                    "ci95_high_pp": 100*hi if np.isfinite(hi) else np.nan,
                    "p_value_block": p,
                })
tails = pd.DataFrame(tail_rows)
if not tails.empty:
    tails["q_value_BH"] = np.nan
    for _, idx in tails.groupby(["side", "threshold_pct", "horizon_bars"]).groups.items():
        tails.loc[idx, "q_value_BH"] = benjamini_hochberg(tails.loc[idx, "p_value_block"].to_numpy())
    tails["fdr_5pct"] = tails["q_value_BH"] < 0.05

# Append the tail probabilities to the SAME compact output file, using a row_type field.
# Keep numeric metric rows and probability rows in one CSV to avoid creating many files.
metric_export = summary.copy()
metric_export.insert(0, "row_type", "metric")
tail_export = tails.copy()
tail_export.insert(0, "row_type", "tail_probability")
# Union columns intentionally creates blanks where a column is not applicable.
combined = pd.concat([metric_export, tail_export], ignore_index=True, sort=False)
combined.to_csv(OUTPUT_FILE, index=False)
print(f"Итоговый единый CSV обновлён: {OUTPUT_FILE.resolve()} ({len(combined):,} строк)")
display(tails.sort_values(
    ["ticker", "side", "threshold_pct", "horizon_bars", "probability_diff_pp"],
    ascending=[True, True, True, True, False]
).head(30))

Итоговый единый CSV обновлён: C:\Users\Сергей Кравцов\SC_03_robust_event_study_summary.csv (11,232 строк)


,ticker,group_type,group,quantile,horizon_bars,threshold_pct,side,episodes_matched,event_probability_pct,control_probability_pct,probability_diff_pp,ci95_low_pp,ci95_high_pp,p_value_block,q_value_BH,fdr_5pct
1369,GAZP,source_family,orderstats::orders_depth,0.99,1,0.5,down,28,3.571429,0.0,3.571429,0.0,10.714286,0.594406,1.0,False
1081,GAZP,source_family,orderstats::imbalance_pressure,0.99,1,0.5,down,39,2.564103,0.0,2.564103,0.0,7.692308,0.589411,1.0,False
865,GAZP,source_family,orderstats::activity_volume,0.95,1,0.5,down,51,1.960784,0.0,1.960784,0.0,5.882353,0.462537,1.0,False
1,GAZP,source,obstats,0.95,1,0.5,down,41,0.000000,0.0,0.000000,0.0,0.000000,1.000000,1.0,False
73,GAZP,source,obstats,0.99,1,0.5,down,24,0.000000,0.0,0.000000,0.0,0.000000,1.000000,1.0,False
145,GAZP,source,orderstats,0.95,1,0.5,down,64,0.000000,0.0,0.000000,0.0,0.000000,1.000000,1.0,False
217,GAZP,source,orderstats,0.99,1,0.5,down,29,0.000000,0.0,0.000000,0.0,0.000000,1.000000,1.0,False
289,GAZP,source,tradestats,0.95,1,0.5,down,60,0.000000,0.0,0.000000,0.0,0.000000,1.000000,1.0,False
361,GAZP,source,tradestats,0.99,1,0.5,down,24,0.000000,0.0,0.000000,0.0,0.000000,1.000000,1.0,False
433,GAZP,source_family,obstats::activity_volume,0.95,1,0.5,down,39,0.000000,0.0,0.000000,0.0,0.000000,1.000000,1.0,False


## 10. Как читать результат и на что смотреть в первую очередь

1. **Для тезиса о крупных скачках** сначала смотри строки `row_type = tail_probability`, `side = either`, пороги 1.5% и 2%, горизонты 5 и 20.
2. Сравни `event_probability_pct` с `control_probability_pct`. Например, рост с 8% до 15% — это +7 процентных пунктов вероятности крупного хода, а не +7% доходности.
3. Проверь `probability_diff_pp`, интервал `ci95_low_pp`–`ci95_high_pp`, `q_value_BH` и `episodes_matched`. Большая разница на 8 эпизодах гораздо менее надёжна, чем похожая разница на 80 эпизодах.
4. Для направления смотри `side = up` и `side = down` отдельно. Если обе стороны дают эффект, это может быть индикатором будущей волатильности, а не направления.
5. В `row_type = metric` показатель `max_abs_move` измеряется в процентах от цены события; `mean_diff` — разница между событием и контролем в процентных пунктах. Для `ret` знак важен, для `abs_ret`, `max_abs_move` и `range` — нет.
6. Сначала оцени MTLR по размеру эффекта и частоте крупных ходов, но не выбирай его только по самой большой цифре. Нужны контрольная разница, неопределённость, достаточное число эпизодов и повторяемость на новых датах.
7. `fdr_5pct=True` означает, что результат прошёл ориентировочную поправку Benjamini–Hochberg в своей семье проверок. Это не гарантирует прибыльность.
8. Следующий шаг после этого исследования — только для заранее выбранных кандидатов: хронологическая проверка на отложенном периоде, затем торговая симуляция с входом не раньше следующей доступной цены, bid/ask, комиссиями, проскальзыванием и правилами выхода. Для OHLC-пути нужно отдельно решить неоднозначность, когда TP и SL могли быть задеты в одном баре.

**Методологическая оговорка:** локальный matching снижает проблему сравнения разных рыночных режимов, а объединение сигналов в неперекрывающиеся эпизоды снижает повторный учёт одних и тех же движений. Однако блоковый bootstrap по эпизодам — приближение, а подбор групп после просмотра результата создаёт риск selection bias. Этот notebook — этап исследования гипотез, не подтверждение готового торгового сигнала.